# Kronos Foundation Models: Zero-Shot Forecasting & Walk-Forward Evaluation

This notebook evaluates Kronos foundation models (base, small, mini) for time-series forecasting in FX markets using a production-grade walk-forward validation framework.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Step 1: Mount Google Drive

Access Google Drive to download the FX-AlphaLab project archive.

In [2]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/FX-AlphaLab.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/FX-AlphaLab')

print("✓ Unzipped successfully")
print("Contents:")
for item in os.listdir('/content/FX-AlphaLab'):
    print(f"  {item}")

✓ Unzipped successfully
Contents:
  FX-AlphaLab


## Step 2: Extract Project Archive

Unzip the FX-AlphaLab project from Google Drive to the Colab compute environment.

In [3]:
import os

for item in os.listdir('/content/FX-AlphaLab/FX-AlphaLab'):
    print(item)

AGENTS.md
economic_events.csv
04a_events_eda (1).ipynb
logs
src
scripts
.env.example
data
datasets
pyproject.toml
.venv-1
.dvcignore
fx_alphalab.egg-info
.gitignore
.ruff_cache
.venv
docs
docker-compose.yml
config
.dvc
.env
src_technical
tests
.pytest_cache
.pre-commit-config.yaml
data.dvc
.git
notebooks
README.md
outputs
.vscode
frontend
.github
CLAUDE.md
models


## Step 3: Verify Project Contents

List the top-level structure of the extracted project.

In [4]:
import sys
sys.path.append('/content/FX-AlphaLab/FX-AlphaLab')

from src.agents.technical.features import (
    load_pair, add_features, get_feature_names, PAIRS, TIMEFRAMES
)
print("✓ features.py imported")
print(f"  Pairs     : {PAIRS}")
print(f"  Timeframes: {TIMEFRAMES}")
print(f"  Features  : {get_feature_names()}")

✓ features.py imported
  Pairs     : ['EURUSDm', 'GBPUSDm', 'USDJPYm', 'USDCHFm']
  Timeframes: ['D1', 'H4', 'H1']
  Features  : ['log_ret', 'log_ret_sq', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_lower', 'bb_pct', 'atr', 'ema_20', 'ema_50', 'ema_200', 'mom_5', 'mom_20']


## Step 4: Import Feature Engineering Module

Load the technical agent's feature engineering utilities to see available currency pairs and timeframes.

In [5]:
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


## Step 5: Verify GPU Availability

Check PyTorch installation and GPU availability for accelerated model inference.

In [7]:
!pip install pyarrow scikit-optimize hmmlearn
!pip install torch --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 12.9 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118


## Step 6: Install Required Dependencies

Install packages for hyperparameter optimization, model interpretation, and deep learning.

In [8]:
import sys
sys.path.append('/content/FXAlphaLab/FX-AlphaLab')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from skopt import BayesSearchCV
import xgboost as xgb
import shap
import joblib

from src.agents.technical.features import (
    load_pair, add_features, get_feature_names, PAIRS, TIMEFRAMES
)

MODEL_DIR = Path("/content/FXAlphaLab/FX-AlphaLab/models/trained")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

data = {}
for pair in PAIRS:
    data[pair] = {}
    for tf in TIMEFRAMES:
        df = load_pair(pair, tf)
        df = add_features(df)
        data[pair][tf] = df

print("✓ Imports OK")
print("✓ Data loaded successfully")
print(f"  Datasets: {sum(len(v) for v in data.values())}")

✓ Imports OK
✓ Data loaded successfully
  Datasets: 12


## Step 7: Load Feature Data

Load OHLCV data for all currency pairs and timeframes, then compute engineered features for baseline model analysis.

In [9]:
# ============================================================
# CELL B1 — Walk-Forward Infrastructure
# ============================================================
import time
import random
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Callable, List, Tuple
from pathlib import Path

# ── Constants ──────────────────────────────────────────────
OUTER_MIN_TRAIN  = 500
OUTER_TEST_SIZE  = 100
OUTER_PURGE      = 5
OUTER_EMBARGO    = 5

ANNUALIZATION = {"D1": 252, "H4": 252 * 6, "H1": 252 * 24}

MODEL_DIR = Path("/content/FX-AlphaLab/FX-AlphaLab/models/trained")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── FoldResult ─────────────────────────────────────────────
@dataclass
class FoldResult:
    y_true:   np.ndarray
    y_pred:   np.ndarray
    ret_true: np.ndarray
    ret_pred: np.ndarray

# ── Expanding splits ───────────────────────────────────────
def make_expanding_splits(
    n_samples: int,
    min_train_size: int,
    test_size: int,
    purge_size: int = 5,
    embargo_size: int = 5,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    splits = []
    train_end = min_train_size
    while train_end + purge_size + test_size <= n_samples:
        test_start = train_end + purge_size
        test_end   = min(test_start + test_size, n_samples)
        train_idx  = np.arange(max(0, train_end - min_train_size), train_end - embargo_size)
        test_idx   = np.arange(test_start, test_end)
        if len(test_idx) >= 10:
            splits.append((train_idx, test_idx))
        train_end += test_size
    return splits

# ── Metrics ────────────────────────────────────────────────
def compute_metrics(folds: List[FoldResult], periods_per_year: int = 252) -> dict:
    if not folds:
        return {"hit_ratio": 0.5, "sharpe": 0.0, "rmse": np.nan, "mape": np.nan, "n_obs": 0}

    y_true_all   = np.concatenate([f.y_true   for f in folds])
    y_pred_all   = np.concatenate([f.y_pred   for f in folds])
    ret_pred_all = np.concatenate([f.ret_pred for f in folds])

    hit_ratio = float(np.mean(y_true_all == y_pred_all))
    sharpe    = (float(ret_pred_all.mean() / ret_pred_all.std() * np.sqrt(periods_per_year))
                 if ret_pred_all.std() > 1e-10 else 0.0)
    rmse      = float(np.sqrt(np.mean((y_true_all - y_pred_all) ** 2)))
    # ✅ MAPE = directional error rate (0–1), NOT price MAPE (avoids Inf)
    mape      = float(np.mean(y_true_all != y_pred_all))

    return {"hit_ratio": hit_ratio, "sharpe": sharpe, "rmse": rmse,
            "mape": mape, "n_obs": len(y_true_all)}

# ── Core walk-forward evaluator for close-price predictors ─
def evaluate_close_model_walkforward(
    df: pd.DataFrame,
    tf: str,
    close_predictor: Callable[[np.ndarray, np.ndarray], np.ndarray],
    min_train_size: int = OUTER_MIN_TRAIN,
    test_size: int      = OUTER_TEST_SIZE,
    purge_size: int     = OUTER_PURGE,
    embargo_size: int   = OUTER_EMBARGO,
) -> Tuple[List[FoldResult], list]:

    close    = df["close"].to_numpy(dtype=np.float32)
    y        = df["target"].to_numpy().astype(int)
    next_ret = df["log_ret"].shift(-1).fillna(0.0).to_numpy(dtype=np.float32)
    n        = len(df)

    # Graceful fallback for short series
    if n < min_train_size + test_size:
        min_train_size = max(200, int(n * 0.6))
        test_size      = max(40,  int(n * 0.2))

    splits = make_expanding_splits(n, min_train_size, test_size, purge_size, embargo_size)
    fold_results = []

    for train_idx, test_idx in splits:
        y_pred  = close_predictor(close[train_idx], close[test_idx]).astype(int)
        y_test  = y[test_idx]
        rt      = next_ret[test_idx]
        rp      = np.array([(p * 2 - 1) * abs(rt[i]) for i, p in enumerate(y_pred)],
                           dtype=np.float32)
        fold_results.append(FoldResult(y_true=y_test, y_pred=y_pred,
                                       ret_true=rt, ret_pred=rp))

    return fold_results, splits

print("✓ Walk-forward infrastructure ready")
print(f"  Min train={OUTER_MIN_TRAIN} | Test={OUTER_TEST_SIZE} | "
      f"Purge={OUTER_PURGE} | Embargo={OUTER_EMBARGO}")

✓ Walk-forward infrastructure ready
  Min train=500 | Test=100 | Purge=5 | Embargo=5


## Step 8: Build Walk-Forward Backtesting Framework

Implement production-grade expanding-window cross-validation with purging and embargo to prevent data leakage. Compute metrics: hit ratio, Sharpe ratio, RMSE, and error rate.

In [13]:
!pip install git+https://github.com/shiyu-coder/Kronos.git

  Cloning https://github.com/shiyu-coder/Kronos.git to /tmp/pip-req-build-0iitto8o
  Running command git clone --filter=blob:none --quiet https://github.com/shiyu-coder/Kronos.git /tmp/pip-req-build-0iitto8o
  Resolved https://github.com/shiyu-coder/Kronos.git to commit 67b630e67f6a18c9e9be918d9b4337c960db1e9a
ERROR: git+https://github.com/shiyu-coder/Kronos.git does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## Step 9: Quick Kronos Installation

Install Kronos directly from GitHub repository.

In [15]:
# ============================================================
# KRONOS INSTALLATION & IMPORT (Colab Fixed Version)
# ============================================================

import os
import sys
from pathlib import Path

# 1. Clone the repository (if not already cloned)
repo_path = Path("/content/Kronos")
if not repo_path.exists():
    print("Cloning Kronos repository...")
    !git clone https://github.com/shiyu-coder/Kronos.git
else:
    print("Kronos repository already exists.")

# 2. Install in editable mode
print("Installing Kronos in editable mode...")
%cd /content/Kronos
!pip install -e . --quiet

# 3. Add to Python path (important for Colab)
sys.path.append("/content/Kronos")

# 4. Import the official classes
try:
    from model import Kronos, KronosTokenizer, KronosPredictor
    print("✅ Kronos imported successfully!")
    print(f"   Kronos classes available: Kronos, KronosTokenizer, KronosPredictor")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Try restarting the Colab runtime and running this cell again.")

# 5. Go back to root folder
%cd /content

print("\nInstallation complete. You can now use Kronos.")

Cloning Kronos repository...
Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 371 (delta 5), reused 4 (delta 4), pack-reused 363 (from 2)
Receiving objects: 100% (371/371), 9.32 MiB | 15.52 MiB/s, done.
Resolving deltas: 100% (174/174), done.
Installing Kronos in editable mode...
/content/Kronos
ERROR: file:///content/Kronos does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
✅ Kronos imported successfully!
   Kronos classes available: Kronos, KronosTokenizer, KronosPredictor
/content

Installation complete. You can now use Kronos.


## Step 10: Install & Configure Kronos Models

Clone, install, and configure the Kronos foundation models with GPU acceleration support and proper Python path setup for Colab.

In [24]:
# ============================================================
# CELL B8 — KRONOS ZERO-SHOT EVALUATION (Correct Timestamp API)
# ============================================================
#
# Signature confirmed:
#   predict(df, x_timestamp, y_timestamp, pred_len,
#           T=1.0, top_k=0, top_p=0.9, sample_count=1, verbose=True)
#
# Fix: x_timestamp and y_timestamp must be pd.Series, not lists.
#      Kronos calls .dt on them internally → plain lists crash.
# ============================================================

import sys, time, warnings
import numpy as np
import pandas as pd
import torch
from typing import Callable, List, Tuple

warnings.filterwarnings("ignore", category=UserWarning)

from model import Kronos, KronosTokenizer, KronosPredictor
print("✅ Kronos imported successfully")

# ── Config ────────────────────────────────────────────────────────────────────
KRONOS_MODELS      = ["NeoQuasar/Kronos-base"]
KRONOS_CONTEXT_LEN = 64
KRONOS_PRED_LEN    = 1
KRONOS_DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
KRONOS_DTYPE       = torch.bfloat16 if torch.cuda.is_available() else torch.float32
TF_TO_OFFSET       = {"D1": "B", "H4": "4h", "H1": "1h"}

print(f"Device : {KRONOS_DEVICE} | dtype: {KRONOS_DTYPE} | Context: {KRONOS_CONTEXT_LEN} bars")

# ── Model cache ───────────────────────────────────────────────────────────────
_kronos_cache = {}

def get_kronos_predictor(model_id: str) -> KronosPredictor:
    if model_id not in _kronos_cache:
        print(f"  Loading {model_id} ... ", end="", flush=True)
        t0        = time.perf_counter()
        model     = Kronos.from_pretrained(model_id).to(KRONOS_DEVICE)
        if KRONOS_DTYPE != torch.float32:
            model = model.to(KRONOS_DTYPE)
        model.eval()
        tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
        predictor = KronosPredictor(model, tokenizer, max_context=KRONOS_CONTEXT_LEN)
        _kronos_cache[model_id] = predictor
        print(f"done ({time.perf_counter()-t0:.1f}s)")
    return _kronos_cache[model_id]

# ── OHLCVA DataFrame builder ──────────────────────────────────────────────────
def build_ohlcva_df(df_slice: pd.DataFrame) -> pd.DataFrame:
    """DatetimeIndex DataFrame, 6 cols [open,high,low,close,volume,amount].
    FX has no real volume/amount → zero-filled (Kronos paper §4.1)."""
    out = pd.DataFrame(index=df_slice.index)
    out["open"]   = df_slice["open"].astype(np.float32)
    out["high"]   = df_slice["high"].astype(np.float32)
    out["low"]    = df_slice["low"].astype(np.float32)
    out["close"]  = df_slice["close"].astype(np.float32)
    out["volume"] = np.float32(0.0)
    out["amount"] = np.float32(0.0)
    return out

def next_timestamp(last_ts: pd.Timestamp, tf: str) -> pd.Timestamp:
    return last_ts + pd.tseries.frequencies.to_offset(TF_TO_OFFSET[tf])

# ── One-step forecast — FIXED: pd.Series for timestamps ──────────────────────
def kronos_forecast_next_close(
    predictor:  KronosPredictor,
    ohlcva_df:  pd.DataFrame,
    tf:         str,
) -> float:
    last_close = float(ohlcva_df["close"].iloc[-1])
    try:
        ctx_window = ohlcva_df.iloc[-KRONOS_CONTEXT_LEN:]

        # ✅ FIX: pd.Series — Kronos calls .dt internally, plain list crashes
        x_timestamp = pd.Series(ctx_window.index)
        y_timestamp = pd.Series([next_timestamp(ctx_window.index[-1], tf)])

        with torch.no_grad():
            output = predictor.predict(
                df          = ctx_window,
                x_timestamp = x_timestamp,   # ← pd.Series, not list
                y_timestamp = y_timestamp,   # ← pd.Series, not list
                pred_len    = KRONOS_PRED_LEN,
                sample_count= 1,
                verbose     = False,
            )

        # ── Parse output ──────────────────────────────────────────────────────
        if isinstance(output, pd.DataFrame):
            pred_close = (float(output["close"].iloc[0])
                          if "close" in output.columns
                          else float(output.iloc[0, 3]))

        elif isinstance(output, (list, tuple)):
            first = output[0]
            if isinstance(first, pd.DataFrame):
                pred_close = (float(first["close"].iloc[0])
                              if "close" in first.columns
                              else float(first.iloc[0, 3]))
            else:
                arr = np.asarray(first, dtype=np.float32).flatten()
                pred_close = float(arr[3]) if len(arr) >= 6 else float(arr[0])

        elif torch.is_tensor(output):
            out_np = output.cpu().float().numpy()
            while out_np.ndim > 2:
                out_np = out_np[0]
            pred_close = float(out_np[0, 3]) if out_np.ndim == 2 else float(out_np[3])

        else:
            out_np = np.asarray(output, dtype=np.float32).flatten()
            pred_close = float(out_np[3]) if len(out_np) >= 6 else float(out_np[0])

        return pred_close if (np.isfinite(pred_close) and pred_close > 0) else last_close

    except Exception as exc:
        # Raise immediately so we see the real error instead of silent fallback
        raise RuntimeError(
            f"kronos predict() failed: {type(exc).__name__}: {exc}\n"
            f"  output type: {type(output) if 'output' in dir() else 'never produced'}\n"
            f"  x_timestamp type: {type(x_timestamp)}, sample: {x_timestamp.iloc[:2].tolist()}\n"
            f"  y_timestamp type: {type(y_timestamp)}, value:  {y_timestamp.tolist()}"
        ) from exc

# ── Predictor factory ─────────────────────────────────────────────────────────
def make_kronos_zero_shot_predictor(model_id: str) -> Callable:
    kp = get_kronos_predictor(model_id)

    def predictor_func(
        train_df: pd.DataFrame,
        test_df:  pd.DataFrame,
        tf:       str,
    ) -> np.ndarray:
        full_ohlcva = build_ohlcva_df(pd.concat([train_df, test_df]))
        train_end   = len(train_df)
        y_hat       = np.zeros(len(test_df), dtype=int)

        for i in range(len(test_df)):
            ctx        = full_ohlcva.iloc[: train_end + i]
            last_c     = float(ctx["close"].iloc[-1])
            pred_c     = kronos_forecast_next_close(kp, ctx, tf)
            y_hat[i]   = 1 if pred_c > last_c else 0

        return y_hat

    return predictor_func

# ── Walk-forward evaluator ────────────────────────────────────────────────────
def evaluate_kronos_walkforward(
    df:               pd.DataFrame,
    tf:               str,
    kronos_predictor: Callable,
    min_train_size:   int = OUTER_MIN_TRAIN,
    test_size:        int = OUTER_TEST_SIZE,
    purge_size:       int = OUTER_PURGE,
    embargo_size:     int = OUTER_EMBARGO,
) -> Tuple[List[FoldResult], list]:

    y        = df["target"].to_numpy().astype(int)
    next_ret = df["log_ret"].shift(-1).fillna(0.0).to_numpy(dtype=np.float32)
    n        = len(df)

    if n < min_train_size + test_size:
        min_train_size = max(200, int(n * 0.6))
        test_size      = max(40,  int(n * 0.2))

    splits       = make_expanding_splits(n, min_train_size, test_size, purge_size, embargo_size)
    fold_results = []

    for train_idx, test_idx in splits:
        y_pred = kronos_predictor(
            df.iloc[train_idx], df.iloc[test_idx], tf,
        ).astype(int)
        y_test = y[test_idx]
        rt     = next_ret[test_idx]
        rp     = ((y_pred * 2 - 1) * rt).astype(np.float32)   # ✅ no abs()
        fold_results.append(
            FoldResult(y_true=y_test, y_pred=y_pred, ret_true=rt, ret_pred=rp)
        )

    return fold_results, splits

# ── Run ───────────────────────────────────────────────────────────────────────
kronos_zs_rows = []
total_jobs     = len(KRONOS_MODELS) * len(PAIRS) * len(TIMEFRAMES)
job_no         = 0
run_t0         = time.perf_counter()

print(f"\nStarting Kronos zero-shot: {total_jobs} jobs\n")

for model_id in KRONOS_MODELS:
    model_tag  = model_id.split("/")[-1]
    predictor  = make_kronos_zero_shot_predictor(model_id)
    print(f"=== {model_id} ===")

    for pair in PAIRS:
        for tf in TIMEFRAMES:
            job_no += 1
            t1      = time.perf_counter()
            print(f"  [{job_no:>2}/{total_jobs}]  {pair:<12} {tf:<4} ...",
                  end=" ", flush=True)

            folds, splits = evaluate_kronos_walkforward(
                df=data[pair][tf].copy(), tf=tf, kronos_predictor=predictor,
            )
            m = compute_metrics(folds, periods_per_year=ANNUALIZATION[tf])

            print(f"hit={m['hit_ratio']:.4f}  sharpe={m['sharpe']:+.3f}  "
                  f"rmse={m['rmse']:.4f}  ({time.perf_counter()-t1:.1f}s)")

            kronos_zs_rows.append({
                "model":     f"kronos_zeroshot_{model_tag}",
                "model_id":  model_id,
                "pair":      pair,
                "timeframe": tf,
                "folds":     len(splits),
                **m,
            })

kronos_zs_df = (pd.DataFrame(kronos_zs_rows)
                  .sort_values(["model","pair","timeframe"])
                  .reset_index(drop=True))

print(f"\n✓ Kronos zero-shot completed in {(time.perf_counter()-run_t0)/60:.1f} min")
display(kronos_zs_df)
kronos_zs_df.to_csv("/content/kronos_zero_shot_results.csv", index=False)
print("Results saved → /content/kronos_zero_shot_results.csv")

✅ Kronos imported successfully
Device : cuda | dtype: torch.bfloat16 | Context: 64 bars

Starting Kronos zero-shot: 12 jobs

  Loading NeoQuasar/Kronos-base ... done (2.7s)
=== NeoQuasar/Kronos-base ===
  [ 1/12]  EURUSDm      D1   ... hit=0.5140  sharpe=+0.223  rmse=0.6971  (33.6s)
  [ 2/12]  EURUSDm      H4   ... hit=0.5084  sharpe=+0.117  rmse=0.7011  (247.8s)
  [ 3/12]  EURUSDm      H1   ... hit=0.5028  sharpe=+0.146  rmse=0.7052  (1010.5s)
  [ 4/12]  GBPUSDm      D1   ... hit=0.4920  sharpe=+0.222  rmse=0.7127  (34.4s)
  [ 5/12]  GBPUSDm      H4   ... hit=0.4983  sharpe=-0.328  rmse=0.7083  (249.0s)
  [ 6/12]  GBPUSDm      H1   ... hit=0.4974  sharpe=-0.133  rmse=0.7090  (1005.6s)
  [ 7/12]  USDJPYm      D1   ... hit=0.5060  sharpe=+0.096  rmse=0.7029  (33.4s)
  [ 8/12]  USDJPYm      H4   ... hit=0.5069  sharpe=+0.229  rmse=0.7022  (252.3s)
  [ 9/12]  USDJPYm      H1   ... hit=0.5079  sharpe=-0.086  rmse=0.7015  (1020.8s)
  [10/12]  USDCHFm      D1   ... hit=0.5080  sharpe=+0.687 

,model,model_id,pair,timeframe,folds,hit_ratio,sharpe,rmse,mape,n_obs
0,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,EURUSDm,D1,10,0.514000,0.222553,0.697137,0.486000,1000
1,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,EURUSDm,H1,305,0.502754,0.145646,0.705157,0.497246,30500
2,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,EURUSDm,H4,75,0.508400,0.117133,0.701142,0.491600,7500
3,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,GBPUSDm,D1,10,0.492000,0.221986,0.712741,0.508000,1000
4,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,GBPUSDm,H1,305,0.497377,-0.133476,0.708959,0.502623,30500
5,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,GBPUSDm,H4,75,0.498267,-0.328159,0.708331,0.501733,7500
6,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,USDCHFm,D1,10,0.508000,0.686911,0.701427,0.492000,1000
7,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,USDCHFm,H1,305,0.504787,0.335035,0.703714,0.495213,30500
8,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,USDCHFm,H4,75,0.502400,0.159614,0.705408,0.497600,7500
9,kronos_zeroshot_Kronos-base,NeoQuasar/Kronos-base,USDJPYm,D1,10,0.506000,0.096492,0.702851,0.494000,1000


Results saved → /content/kronos_zero_shot_results.csv


## Step 11: Evaluate Kronos Base Model with Zero-Shot Walk-Forward

Run zero-shot evaluation of Kronos base model across all currency pairs and timeframes using walk-forward validation. Build production helpers for OHLCVA dataframes and timestamp handling.

In [27]:
# ============================================================
# CELL B9 — KRONOS SMALL & MINI ZERO-SHOT EVALUATION
# ============================================================
# Confirmed model zoo (NeoQuasar namespace):
#   NeoQuasar/Kronos-mini   ← smallest / fastest
#   NeoQuasar/Kronos-small  ← mid-size
#   NeoQuasar/Kronos-base   ← already done in Cell B8
#
# "Kronos-large" does not exist on HuggingFace as of now.
# ============================================================

import time
import pandas as pd

KRONOS_REMAINING_MODELS = [
    "NeoQuasar/Kronos-small",
    "NeoQuasar/Kronos-mini",
]

# Skip H1 to keep runtime reasonable — flip to True to include
RUN_H1 = False
timeframes_run = [tf for tf in TIMEFRAMES if tf != "H1"] if not RUN_H1 else TIMEFRAMES

if not RUN_H1:
    print(f"⚠ Skipping H1  →  running: {timeframes_run}\n")

# ── VRAM check ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    free_vram = (torch.cuda.get_device_properties(0).total_memory
                 - torch.cuda.memory_allocated()) / 1e9
    print(f"Free VRAM: {free_vram:.1f} GB\n")

# ── Run all remaining models ──────────────────────────────────────────────────
remaining_rows = []

for model_id in KRONOS_REMAINING_MODELS:
    model_tag = model_id.split("/")[-1]          # e.g. "Kronos-small"
    predictor = make_kronos_zero_shot_predictor(model_id)   # uses Cell B8 cache + loader

    total_jobs = len(PAIRS) * len(timeframes_run)
    job_no     = 0
    t_model    = time.perf_counter()

    print(f"\n=== {model_id} ===  ({total_jobs} jobs)")

    for pair in PAIRS:
        for tf in timeframes_run:
            job_no += 1
            t1      = time.perf_counter()
            print(f"  [{job_no:>2}/{total_jobs}]  {pair:<12} {tf:<4} ...",
                  end=" ", flush=True)

            folds, splits = evaluate_kronos_walkforward(
                df               = data[pair][tf].copy(),
                tf               = tf,
                kronos_predictor = predictor,
            )
            m = compute_metrics(folds, periods_per_year=ANNUALIZATION[tf])

            print(f"hit={m['hit_ratio']:.4f}  sharpe={m['sharpe']:+.3f}  "
                  f"rmse={m['rmse']:.4f}  ({time.perf_counter()-t1:.1f}s)")

            remaining_rows.append({
                "model":     f"kronos_zeroshot_{model_tag}",
                "model_id":  model_id,
                "pair":      pair,
                "timeframe": tf,
                "folds":     len(splits),
                **m,
            })

    print(f"  → {model_tag} done in {(time.perf_counter()-t_model)/60:.1f} min")

# ── Assemble results ──────────────────────────────────────────────────────────
remaining_df = (pd.DataFrame(remaining_rows)
                  .sort_values(["model", "pair", "timeframe"])
                  .reset_index(drop=True))

display(remaining_df)
remaining_df.to_csv("/content/kronos_small_mini_results.csv", index=False)
print("Saved → /content/kronos_small_mini_results.csv")

# ── Full Kronos family comparison ─────────────────────────────────────────────
print("\n── Full Kronos family comparison (avg over completed datasets) ──")

try:
    base_df = pd.read_csv("/content/kronos_zero_shot_results.csv")
    # Filter base to same timeframes for fair comparison
    base_sub = base_df[base_df["timeframe"].isin(timeframes_run)]
    all_kronos = pd.concat([base_sub, remaining_df], ignore_index=True)

    summary = (
        all_kronos
        .groupby("model", as_index=False)
        .agg(
            hit_ratio = ("hit_ratio", "mean"),
            sharpe    = ("sharpe",    "mean"),
            rmse      = ("rmse",      "mean"),
            n_obs     = ("n_obs",     "sum"),
        )
        .sort_values("sharpe", ascending=False)
        .reset_index(drop=True)
    )
    summary["hit_ratio"] = summary["hit_ratio"].map("{:.4f}".format)
    summary["sharpe"]    = summary["sharpe"].map("{:+.3f}".format)
    summary["rmse"]      = summary["rmse"].map("{:.4f}".format)
    display(summary)

    # Save combined
    all_kronos.to_csv("/content/kronos_all_models_results.csv", index=False)
    print("Combined saved → /content/kronos_all_models_results.csv")

except FileNotFoundError:
    print("  (kronos_zero_shot_results.csv not found — run Cell B8 first)")

⚠ Skipping H1  →  running: ['D1', 'H4']

Free VRAM: 15.2 GB

  Loading NeoQuasar/Kronos-small ... 

config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

done (2.8s)

=== NeoQuasar/Kronos-small ===  (8 jobs)
  [ 1/8]  EURUSDm      D1   ... hit=0.5110  sharpe=+0.409  rmse=0.6993  (28.1s)
  [ 2/8]  EURUSDm      H4   ... hit=0.5005  sharpe=+0.039  rmse=0.7067  (211.1s)
  [ 3/8]  GBPUSDm      D1   ... hit=0.4770  sharpe=-0.394  rmse=0.7232  (28.9s)
  [ 4/8]  GBPUSDm      H4   ... hit=0.4920  sharpe=-0.228  rmse=0.7127  (212.3s)
  [ 5/8]  USDJPYm      D1   ... hit=0.4820  sharpe=+0.137  rmse=0.7197  (28.0s)
  [ 6/8]  USDJPYm      H4   ... hit=0.4912  sharpe=-0.336  rmse=0.7133  (210.7s)
  [ 7/8]  USDCHFm      D1   ... hit=0.4900  sharpe=-0.656  rmse=0.7141  (27.9s)
  [ 8/8]  USDCHFm      H4   ... hit=0.4969  sharpe=-0.706  rmse=0.7093  (211.3s)
  → Kronos-small done in 16.0 min
  Loading NeoQuasar/Kronos-mini ... 

config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

...

done (1.8s)

=== NeoQuasar/Kronos-mini ===  (8 jobs)
  [ 1/8]  EURUSDm      D1   ... hit=0.5110  sharpe=-0.407  rmse=0.6993  (23.6s)
  [ 2/8]  EURUSDm      H4   ... hit=0.5025  sharpe=-0.185  rmse=0.7053  (172.3s)
  [ 3/8]  GBPUSDm      D1   ... hit=0.5080  sharpe=-0.163  rmse=0.7014  (23.5s)
  [ 4/8]  GBPUSDm      H4   ... hit=0.5113  sharpe=-0.172  rmse=0.6990  (173.3s)
  [ 5/8]  USDJPYm      D1   ... hit=0.5100  sharpe=+0.476  rmse=0.7000  (23.1s)
  [ 6/8]  USDJPYm      H4   ... hit=0.4985  sharpe=-0.431  rmse=0.7081  (168.8s)
  [ 7/8]  USDCHFm      D1   ... hit=0.4980  sharpe=-0.712  rmse=0.7085  (23.2s)
  [ 8/8]  USDCHFm      H4   ... hit=0.4936  sharpe=-0.481  rmse=0.7116  (168.6s)
  → Kronos-mini done in 12.9 min


,model,model_id,pair,timeframe,folds,hit_ratio,sharpe,rmse,mape,n_obs
0,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,EURUSDm,D1,10,0.511000,-0.406732,0.699285,0.489000,1000
1,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,EURUSDm,H4,75,0.502533,-0.185320,0.705313,0.497467,7500
2,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,GBPUSDm,D1,10,0.508000,-0.162937,0.701427,0.492000,1000
3,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,GBPUSDm,H4,75,0.511333,-0.172251,0.699047,0.488667,7500
4,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,USDCHFm,D1,10,0.498000,-0.711928,0.708520,0.502000,1000
5,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,USDCHFm,H4,75,0.493600,-0.480642,0.711618,0.506400,7500
6,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,USDJPYm,D1,10,0.510000,0.475940,0.700000,0.490000,1000
7,kronos_zeroshot_Kronos-mini,NeoQuasar/Kronos-mini,USDJPYm,H4,75,0.498533,-0.430597,0.708143,0.501467,7500
8,kronos_zeroshot_Kronos-small,NeoQuasar/Kronos-small,EURUSDm,D1,10,0.511000,0.408716,0.699285,0.489000,1000
9,kronos_zeroshot_Kronos-small,NeoQuasar/Kronos-small,EURUSDm,H4,75,0.500533,0.038957,0.706730,0.499467,7500


Saved → /content/kronos_small_mini_results.csv

── Full Kronos family comparison (avg over completed datasets) ──


,model,hit_ratio,sharpe,rmse,n_obs
0,kronos_zeroshot_Kronos-base,0.5045,+0.176,0.7039,34000
1,kronos_zeroshot_Kronos-small,0.4926,-0.217,0.7123,34000
2,kronos_zeroshot_Kronos-mini,0.5041,-0.259,0.7042,34000


Combined saved → /content/kronos_all_models_results.csv


## Step 12: Evaluate Kronos Small & Mini Models with Zero-Shot Walk-Forward

Run zero-shot evaluation of Kronos small and mini models across selected currency pairs and timeframes. Aggregate results across all Kronos model sizes for comparative analysis.